In [2]:
!pip install pennylane
!pip install torch
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 65.4 MB/s eta 0:00:00


In [3]:
# ==========================================
# 1. Imports
# ==========================================

import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [6]:
# ==========================================
# 2. Load and Prepare Data (ALL 4 features)
# ==========================================

iris = load_iris()
X = iris.data
y = iris.target

# Normalize
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Map to (0, π)
X = 1 / (1 + np.exp(-X)) * np.pi

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


# ==========================================
# 3. Quantum Reservoir
# ==========================================

num_qubits = 4
reservoir_layers = 3

dev = qml.device("default.qubit", wires=num_qubits)

np.random.seed(42)

reservoir_params_crx = np.random.uniform(
    0, 2*np.pi, size=(reservoir_layers, num_qubits)
).astype(np.float32)

reservoir_params_rx = np.random.uniform(
    0, 2*np.pi, size=(reservoir_layers, num_qubits)
).astype(np.float32)


@qml.qnode(dev, interface="torch")
def quantum_reservoir(x):
    qml.AngleEmbedding(x, wires=range(num_qubits))

    for layer in range(reservoir_layers):
        for i in range(num_qubits):
            qml.CRX(reservoir_params_crx[layer, i], wires=[i, (i + 1) % num_qubits])
        for i in range(num_qubits):
            qml.RX(reservoir_params_rx[layer, i], wires=i)

    # return a list of PennyLane measurements
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]


# ==========================================
# 4. Hybrid Model
# ==========================================

class QRCModel(nn.Module):
    def __init__(self, num_qubits, num_classes):
        super().__init__()
        self.linear = nn.Linear(num_qubits, num_classes)

    def forward(self, x):
        outputs = []
        for sample in x:
            q_out_list = quantum_reservoir(sample)          # list of tensors
            q_out = torch.stack(q_out_list).float()         # stack here
            outputs.append(q_out)

        quantum_batch = torch.stack(outputs)
        return self.linear(quantum_batch)


model = QRCModel(num_qubits=num_qubits, num_classes=3)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


# ==========================================
# 5. Training
# ==========================================

epochs = 200

for epoch in range(epochs):

    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}, Loss: {loss.item():.4f}")


# ==========================================
# 6. Evaluation Metrics
# ==========================================

model.eval()
with torch.no_grad():
    train_pred = torch.argmax(model(X_train), dim=1)
    test_pred  = torch.argmax(model(X_test), dim=1)

train_pred_np = train_pred.numpy()
test_pred_np  = test_pred.numpy()

y_train_np = y_train.numpy()
y_test_np  = y_test.numpy()

# Accuracy
train_acc = accuracy_score(y_train_np, train_pred_np)
test_acc  = accuracy_score(y_test_np, test_pred_np)

# Precision & Recall (macro average for multiclass)
precision = precision_score(y_test_np, test_pred_np, average="macro")
recall    = recall_score(y_test_np, test_pred_np, average="macro")

# Confusion matrix
conf_matrix = confusion_matrix(y_test_np, test_pred_np)

print("\n==============================")
print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)
print("Precision     :", precision)
print("Recall        :", recall)
print("\nConfusion Matrix:\n", conf_matrix)
print("==============================")

Epoch   0, Loss: 1.1551
Epoch  20, Loss: 0.9937
Epoch  40, Loss: 0.8989
Epoch  60, Loss: 0.8238
Epoch  80, Loss: 0.7594
Epoch 100, Loss: 0.7039
Epoch 120, Loss: 0.6556
Epoch 140, Loss: 0.6135
Epoch 160, Loss: 0.5767
Epoch 180, Loss: 0.5442

Train Accuracy: 0.9333333333333333
Test Accuracy : 0.9333333333333333
Precision     : 0.9444444444444445
Recall        : 0.9333333333333332

Confusion Matrix:
 [[10  0  0]
 [ 0  8  2]
 [ 0  0 10]]
